#### support request agent

Build and deploy a generic support-request agent with order context and user interaction history.

Outputs include:
- credit recommendation
- refund recommendation
- draft customer response
- past interactions summary
- order details summary

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")
SUPPORT_AGENT_ENDPOINT_NAME = dbutils.widgets.get("SUPPORT_AGENT_ENDPOINT_NAME")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.support")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.ai")

# Bootstrap core support tables so blank workspaces do not fail before downstream stages materialize data.
spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG}.support.support_agent_reports (support_request_id STRING, user_id STRING, order_id STRING, request_text STRING, ts TIMESTAMP, agent_response STRING)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG}.support.raw_support_requests (support_request_id STRING, user_id STRING, order_id STRING, ts TIMESTAMP, request_type STRING, edge_case_type STRING, request_text STRING)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG}.support.support_request_features (support_request_id STRING, user_id STRING, order_id STRING, ts TIMESTAMP, repeat_complaints_30d BIGINT, policy_limit_usd DOUBLE, risk_score DOUBLE)")

# Backfill-safe schema evolution for existing tables.
try:
    spark.sql(f"ALTER TABLE {CATALOG}.support.support_agent_reports ADD COLUMNS (request_text STRING)")
except Exception:
    # Column already exists or table metadata already evolved.
    pass

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ai.get_order_overview(oid STRING)
RETURNS TABLE (
  order_id STRING,
  location STRING,
  items_json STRING,
  customer_address STRING,
  order_created_ts TIMESTAMP
)
RETURN
  SELECT
    order_id,
    location,
    get_json_object(body, '$.items') AS items_json,
    get_json_object(body, '$.customer_addr') AS customer_address,
    try_to_timestamp(ts) AS order_created_ts
  FROM {CATALOG}.lakeflow.all_events
  WHERE order_id = oid AND event_type = 'order_created'
  LIMIT 1
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ai.get_order_timing(oid STRING)
RETURNS TABLE (
  order_id STRING,
  order_created_ts TIMESTAMP,
  delivered_ts TIMESTAMP,
  delivery_duration_minutes FLOAT
)
RETURN
  WITH order_events AS (
    SELECT order_id, event_type, try_to_timestamp(ts) AS event_ts
    FROM {CATALOG}.lakeflow.all_events
    WHERE order_id = oid
  )
  SELECT
    order_id,
    MIN(CASE WHEN event_type='order_created' THEN event_ts END) AS order_created_ts,
    MAX(CASE WHEN event_type='delivered' THEN event_ts END) AS delivered_ts,
    CAST((UNIX_TIMESTAMP(MAX(CASE WHEN event_type='delivered' THEN event_ts END)) - UNIX_TIMESTAMP(MIN(CASE WHEN event_type='order_created' THEN event_ts END))) / 60 AS FLOAT) AS delivery_duration_minutes
  FROM order_events
  GROUP BY order_id
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ai.get_user_support_history(uid STRING)
RETURNS TABLE (
  support_request_id STRING,
  order_id STRING,
  ts TIMESTAMP,
  request_text STRING,
  agent_response STRING
)
RETURN
  SELECT support_request_id, order_id, ts, request_text, agent_response
  FROM {CATALOG}.support.support_agent_reports
  WHERE user_id = uid
  ORDER BY ts DESC
  LIMIT 10
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ai.get_user_request_history(uid STRING)
RETURNS TABLE (
  support_request_id STRING,
  order_id STRING,
  ts TIMESTAMP,
  request_type STRING,
  edge_case_type STRING,
  request_text STRING
)
RETURN
  SELECT support_request_id, order_id, ts, request_type, edge_case_type, request_text
  FROM {CATALOG}.support.raw_support_requests
  WHERE user_id = uid
  ORDER BY ts DESC
  LIMIT 10
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.ai.get_support_features(sid STRING)
RETURNS TABLE (
  support_request_id STRING,
  repeat_complaints_30d BIGINT,
  policy_limit_usd DOUBLE,
  risk_score DOUBLE,
  feature_ts TIMESTAMP
)
RETURN
  SELECT
    support_request_id,
    repeat_complaints_30d,
    policy_limit_usd,
    risk_score,
    ts AS feature_ts
  FROM {CATALOG}.support.support_request_features
  WHERE support_request_id = sid
  LIMIT 1
""")

In [ ]:
%pip install -U -qqqq typing_extensions dspy-ai mlflow unitycatalog-openai[databricks] openai databricks-sdk databricks-agents pydantic
%restart_python

In [ ]:
import mlflow
import sys
from databricks.sdk import WorkspaceClient

CATALOG = dbutils.widgets.get("CATALOG")
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")
SUPPORT_AGENT_ENDPOINT_NAME = dbutils.widgets.get("SUPPORT_AGENT_ENDPOINT_NAME")

sys.path.append('../utils')
from uc_state import add

dev_experiment = mlflow.set_experiment(f"/Shared/{CATALOG}_support_agent_dev")
add(CATALOG, "experiments", {"experiment_id": dev_experiment.experiment_id, "name": dev_experiment.name})

In [ ]:
%%writefile agent.py
import json
import re
import warnings
from typing import Any, Dict, Literal, Optional
from uuid import uuid4

import dspy
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse
from pydantic import BaseModel, Field
from unitycatalog.ai.core.base import get_uc_function_client

warnings.filterwarnings("ignore", message=".*Ignoring the default notebook Spark session.*")

mlflow.dspy.autolog(log_traces=True)

LLM_MODEL = "{{LLM_MODEL}}"
CATALOG = "{{CATALOG}}"

lm = dspy.LM(f"databricks/{LLM_MODEL}", max_tokens=2000)
dspy.configure(lm=lm)
uc_client = get_uc_function_client()


class SupportReport(BaseModel):
    support_request_id: str
    user_id: str
    order_id: str
    credit_recommendation: Optional[dict] = None
    refund_recommendation: Optional[dict] = None
    draft_response: str
    past_interactions_summary: str
    order_details_summary: str
    decision_confidence: Literal["high", "medium", "low"] = Field(default="medium")
    escalation_flag: bool = Field(default=False)


class SupportTriage(dspy.Signature):
    """Produce a final support resolution report using full context.

    Required inputs to reason over:
    - support request text and edge-case metadata
    - order details and delivery timing
    - prior support history for this user
    - support features: repeat_complaints_30d, policy_limit_usd, risk_score

    Decision policy:
    - Severe repeat/high-risk issue: provide concrete refund + credit and apology.
    - Moderate issue: provide bounded partial refund and/or credit with rationale.
    - Minor/non-compensable issue: typically no refund, small goodwill credit or clear denial.
    - Spam/abusive patterns: avoid excessive compensation, set escalation_flag=true.

    Never return placeholder text like "we will look into this".
    Always return a final customer-ready message with concrete next steps.
    """

    support_request: str = dspy.InputField(desc="Structured case payload JSON string")
    support_request_id: str = dspy.OutputField()
    user_id: str = dspy.OutputField()
    order_id: str = dspy.OutputField()
    credit_recommendation_json: str = dspy.OutputField(desc="JSON object with amount_usd/reason")
    refund_recommendation_json: str = dspy.OutputField(desc="JSON object with amount_usd/reason")
    draft_response: str = dspy.OutputField()
    past_interactions_summary: str = dspy.OutputField()
    order_details_summary: str = dspy.OutputField()
    decision_confidence: str = dspy.OutputField(desc="high, medium, low")
    escalation_flag: str = dspy.OutputField(desc="true or false")


def get_order_overview(order_id: str) -> str:
    return str(uc_client.execute_function(f"{CATALOG}.ai.get_order_overview", {"oid": order_id}).value)


def get_order_timing(order_id: str) -> str:
    return str(uc_client.execute_function(f"{CATALOG}.ai.get_order_timing", {"oid": order_id}).value)


def get_user_support_history(user_id: str) -> str:
    return str(uc_client.execute_function(f"{CATALOG}.ai.get_user_support_history", {"uid": user_id}).value)


def get_user_request_history(user_id: str) -> str:
    return str(uc_client.execute_function(f"{CATALOG}.ai.get_user_request_history", {"uid": user_id}).value)


def get_support_features(support_request_id: str) -> str:
    return str(uc_client.execute_function(f"{CATALOG}.ai.get_support_features", {"sid": support_request_id}).value)


def _parse_case_payload(raw: str) -> Dict[str, Any]:
    try:
        payload = json.loads(raw)
        return payload if isinstance(payload, dict) else {}
    except Exception:
        parsed: Dict[str, Any] = {}
        for key in ["support_request_id", "user_id", "order_id", "text"]:
            match = re.search(rf"{key}=([^\s]+)", raw)
            if match:
                parsed[key] = match.group(1)
        if "text" in parsed:
            parsed["request_text"] = parsed["text"]
        return parsed


def _safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except Exception:
        return default


def _safe_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except Exception:
        return default


def _compute_policy_resolution(payload: Dict[str, Any]) -> Dict[str, Any]:
    features = payload.get("feature_context", {}) if isinstance(payload.get("feature_context"), dict) else {}
    request_text = str(payload.get("request_text", ""))
    edge_case_type = str(payload.get("edge_case_type", "standard"))

    repeat_count = _safe_int(features.get("repeat_complaints_30d"), 0)
    risk_score = _safe_float(features.get("risk_score"), 0.25)
    policy_limit = max(0.0, _safe_float(features.get("policy_limit_usd"), 20.0))

    text_lower = request_text.lower()
    mentions_missing = "missing" in text_lower
    mentions_delay = "late" in text_lower or "delay" in text_lower
    abusive = edge_case_type == "abusive_language"
    spammy = edge_case_type == "spammy_escalation"

    if abusive or (spammy and repeat_count >= 4 and risk_score < 0.4):
        refund_amount = 0.0
        credit_amount = min(3.0, policy_limit)
        escalation = True
        confidence = "medium"
        reason = "Abusive/spammy pattern detected; restricted compensation with escalation."
    elif repeat_count >= 2 or risk_score >= 0.75 or (mentions_missing and mentions_delay):
        refund_amount = round(min(policy_limit, max(policy_limit * 0.9, 15.0)), 2)
        credit_amount = round(min(10.0, max(5.0, policy_limit * 0.25)), 2)
        escalation = False
        confidence = "high"
        reason = "Repeated or high-risk service failure."
    elif repeat_count == 1 or risk_score >= 0.45 or mentions_missing or mentions_delay:
        refund_amount = round(min(policy_limit, max(policy_limit * 0.5, 6.0)), 2)
        credit_amount = round(min(7.0, max(3.0, policy_limit * 0.15)), 2)
        escalation = False
        confidence = "high"
        reason = "Moderate service issue with customer impact."
    else:
        refund_amount = 0.0
        credit_amount = round(min(5.0, max(2.0, policy_limit * 0.1)), 2)
        escalation = False
        confidence = "medium"
        reason = "Minor issue or low-confidence compensable case."

    return {
        "refund_amount": refund_amount,
        "credit_amount": credit_amount,
        "escalation": escalation,
        "confidence": confidence,
        "reason": reason,
        "repeat_count": repeat_count,
    }


def _ordinal(value: int) -> str:
    if 10 <= value % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(value % 10, "th")
    return f"{value}{suffix}"


def _compose_customer_message(payload: Dict[str, Any], policy: Dict[str, Any]) -> str:
    request_text = str(payload.get("request_text", "")).strip()
    repeat_count = policy["repeat_count"]
    refund_amount = policy["refund_amount"]
    credit_amount = policy["credit_amount"]

    lead = "Thanks for contacting us, and I am really sorry about this experience."
    if repeat_count >= 2:
        lead = (
            f"I see this is at least the {_ordinal(repeat_count)} time you've had to contact us for similar issues, "
            "and that is not acceptable."
        )

    resolution_parts = []
    if refund_amount > 0:
        resolution_parts.append(f"a ${refund_amount:.2f} refund")
    if credit_amount > 0:
        resolution_parts.append(f"a ${credit_amount:.2f} credit")

    if resolution_parts:
        resolution = "We are offering " + " and ".join(resolution_parts) + " on this case."
    else:
        resolution = "We are escalating this case for manual review and will follow up quickly."

    context_line = f"Issue noted: {request_text}" if request_text else "Issue noted from your support ticket."

    return f"{lead} {resolution} {context_line}"


class SupportModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.react = dspy.ReAct(
            signature=SupportTriage,
            tools=[
                get_order_overview,
                get_order_timing,
                get_user_support_history,
                get_user_request_history,
                get_support_features,
            ],
            max_iters=10,
        )

    def forward(self, support_request: str) -> SupportReport:
        payload = _parse_case_payload(support_request)
        result = self.react(support_request=support_request)

        policy = _compute_policy_resolution(payload)
        final_message = _compose_customer_message(payload, policy)

        # Guarantee concrete recommendations and customer-ready final response.
        credit_recommendation = {
            "amount_usd": policy["credit_amount"],
            "reason": policy["reason"],
        }
        refund_recommendation = {
            "amount_usd": policy["refund_amount"],
            "reason": policy["reason"],
        }

        return SupportReport(
            support_request_id=str(payload.get("support_request_id") or result.support_request_id),
            user_id=str(payload.get("user_id") or result.user_id),
            order_id=str(payload.get("order_id") or result.order_id),
            credit_recommendation=credit_recommendation,
            refund_recommendation=refund_recommendation,
            draft_response=final_message,
            past_interactions_summary=str(result.past_interactions_summary),
            order_details_summary=str(result.order_details_summary),
            decision_confidence=policy["confidence"],
            escalation_flag=bool(policy["escalation"]),
        )


class DSPySupportAgent(ResponsesAgent):
    def __init__(self):
        self.module = SupportModule()

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        user_message = None
        for msg in request.input:
            msg_dict = msg.model_dump() if hasattr(msg, "model_dump") else msg
            if msg_dict.get("role") == "user":
                user_message = msg_dict.get("content", "")
                break
        if not user_message:
            raise ValueError("No user message found")

        result = self.module(support_request=user_message)
        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text=result.model_dump_json(), id=str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )


AGENT = DSPySupportAgent()
mlflow.models.set_model(AGENT)

In [ ]:
from pathlib import Path

agent_path = Path("agent.py")
source = agent_path.read_text(encoding="utf-8")
source = source.replace("{{LLM_MODEL}}", LLM_MODEL).replace("{{CATALOG}}", CATALOG)
agent_path.write_text(source, encoding="utf-8")

print("Wrote agent.py with concrete CATALOG/LLM_MODEL values")

In [ ]:
import json
import mlflow
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint

sample = spark.sql(f"SELECT support_request_id, user_id, order_id, request_type, edge_case_type, request_text FROM {CATALOG}.support.raw_support_requests LIMIT 1").collect()
if sample:
    row = sample[0]
    payload = {
        "support_request_id": row["support_request_id"],
        "user_id": row["user_id"],
        "order_id": row["order_id"],
        "request_type": row["request_type"],
        "edge_case_type": row["edge_case_type"],
        "request_text": row["request_text"],
        "feature_context": {
            "repeat_complaints_30d": 2,
            "policy_limit_usd": 20.0,
            "risk_score": 0.67,
        },
        "order_context": {},
        "past_request_context": [],
    }
else:
    payload = {
        "support_request_id": "sample-1",
        "user_id": "user-1",
        "order_id": "order-1",
        "request_type": "missing_items",
        "edge_case_type": "repeat_follow_up",
        "request_text": "This is the second time items were missing from my order.",
        "feature_context": {
            "repeat_complaints_30d": 2,
            "policy_limit_usd": 20.0,
            "risk_score": 0.71,
        },
        "order_context": {},
        "past_request_context": [],
    }

resources = [DatabricksServingEndpoint(endpoint_name=LLM_MODEL)]
for fn in [
    f"{CATALOG}.ai.get_order_overview",
    f"{CATALOG}.ai.get_order_timing",
    f"{CATALOG}.ai.get_user_support_history",
    f"{CATALOG}.ai.get_user_request_history",
    f"{CATALOG}.ai.get_support_features",
]:
    resources.append(DatabricksFunction(function_name=fn))

input_example = {"input": [{"role": "user", "content": json.dumps(payload)}]}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="support_agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
    )

mlflow.set_active_model(model_id=logged_agent_info.model_id)
mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{CATALOG}.ai.support_agent"
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

In [ ]:
import mlflow
import time
from databricks import agents
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady

prod_experiment = mlflow.set_experiment(f"/Shared/{CATALOG}_support_agent_prod")
add(CATALOG, "experiments", {"experiment_id": prod_experiment.experiment_id, "name": prod_experiment.name})

deployment_info = None
try:
    deployment_info = agents.deploy(
        model_name=UC_MODEL_NAME,
        model_version=uc_registered_model_info.version,
        scale_to_zero=True,
        endpoint_name=SUPPORT_AGENT_ENDPOINT_NAME,
        environment_vars={"MLFLOW_EXPERIMENT_ID": str(prod_experiment.experiment_id)},
    )
except ValueError as e:
    # Blank-workspace reliability: another run may already be updating this endpoint.
    if "currently updating" not in str(e).lower():
        raise
    print(f"Endpoint {SUPPORT_AGENT_ENDPOINT_NAME} already updating; waiting for readiness.")

workspace = WorkspaceClient()
ready = False
for _ in range(120):
    endpoint = workspace.serving_endpoints.get(name=SUPPORT_AGENT_ENDPOINT_NAME)
    if endpoint.state and endpoint.state.ready == EndpointStateReady.READY:
        ready = True
        break
    time.sleep(10)

if not ready:
    raise RuntimeError(f"Endpoint {SUPPORT_AGENT_ENDPOINT_NAME} did not reach READY within polling window")

if deployment_info is None:
    deployment_info = workspace.serving_endpoints.get(name=SUPPORT_AGENT_ENDPOINT_NAME)

add(CATALOG, "endpoints", deployment_info)
print(f"Endpoint {SUPPORT_AGENT_ENDPOINT_NAME} is READY")